<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/01_consolidar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [1]:
import os, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/proyecto_oro/"
except ImportError:
    BASE_PATH = "./_ci_cache/"

RAW_PATH     = BASE_PATH + "data/raw/"
LANDING_PATH = BASE_PATH + "data/landing/"
os.makedirs(LANDING_PATH, exist_ok=True)


In [2]:
df_oro = pd.read_csv(RAW_PATH + "oro_xauusd.csv", index_col=0, parse_dates=True)
df_cop = pd.read_csv(RAW_PATH + "usd_cop.csv", index_col=0, parse_dates=True)

df = pd.DataFrame(index=df_oro.index)
df['oro_xauusd_Close'] = df_oro['Close']
df = df.join(df_cop['Close'].rename('usd_cop_Close'), how='inner')

df['oro_cop'] = df['oro_xauusd_Close'] * df['usd_cop_Close']

print(f"Filas núcleo (oro + usd_cop): {len(df)}")
df.head()

Filas núcleo (oro + usd_cop): 1423


,oro_xauusd_Close,usd_cop_Close,oro_cop
Date,,,
2021-01-04,182.330002,3420.250000,623614.188763
2021-01-05,182.869995,3447.750000,630490.025665
2021-01-06,179.899994,3442.250000,619260.753990
2021-01-07,179.479996,3412.800049,612529.338183
2021-01-08,173.339996,3488.239990,604651.507133


In [3]:
otras_vars = {
    "dxy":       "dxy.csv",
    "vix":       "vix.csv",
    "wti_crudo": "wti_crudo.csv",
    "bono_10y":  "bono_10y.csv",
}

for nombre, archivo in otras_vars.items():
    data = pd.read_csv(RAW_PATH + archivo, index_col=0, parse_dates=True)
    df[f'{nombre}_Close'] = data['Close']
    print(f"✓ {nombre} agregado")

print(f"\nFilas totales (sin cambiar, siguen basadas en oro+usd_cop): {len(df)}")
print(f"Columnas totales: {len(df.columns)}")

✓ dxy agregado
✓ vix agregado
✓ wti_crudo agregado
✓ bono_10y agregado

Filas totales (sin cambiar, siguen basadas en oro+usd_cop): 1423
Columnas totales: 7


In [4]:
df.index.name = 'DATE'
df_out = df.reset_index()
df_out['DATE'] = df_out['DATE'].astype(str)

df_out.to_csv(LANDING_PATH + "consolidado_oro_cop.csv", index=False)

print(f"✅ Guardado: {len(df_out)} filas × {len(df_out.columns)} columnas")
print(f"📅 {df_out['DATE'].min()} → {df_out['DATE'].max()}")

✅ Guardado: 1423 filas × 8 columnas
📅 2021-01-04 → 2026-09-03
